# Lab 3: Comparing CNNs to Traditional Neural Networks

**AutoParts Inc. — Manufacturing Intelligence Team**

**Context:** A teammate new to deep learning has asked why we're proposing a CNN instead of a fully-connected (FC) network for the automated visual inspection system. This notebook is that explanation, built around our actual use case: classifying parts as defective/non-defective from the 50,000-image production-line dataset, where inputs are full-color images and defects (cracks, deformations, finish issues) can appear anywhere in the frame.

## 1. Parameter Efficiency: A Concrete Comparison

Take one representative image size from our pipeline: **224×224×3** (a typical resolution for a part photographed under the line's fixed camera). Flattened, that's `224 × 224 × 3 = 150,528` input values.

**Fully-connected network:** In an FC network, the first hidden layer connects *every* input pixel to *every* neuron in that layer. With a modest 1,024-neuron hidden layer, the weight matrix alone is `150,528 × 1,024 = 154,140,672` parameters — **~154 million** — before we've done anything except take one step into the network. Double the image resolution to 448×448 and that number nearly quadruples to ~617 million, because FC parameter count scales directly with input size.

**CNN:** A convolutional layer with 32 filters of size 3×3 over 3 input channels has `3 × 3 × 3 × 32 = 864` weights (plus 32 biases = 896 total). Critically, **this number does not depend on image size** — the same 864 weights slide across a 224×224 image or a 4K image; only the amount of compute (not the parameter count) changes.

The code below makes the comparison concrete.

In [1]:
def fc_layer_params(input_h, input_w, channels, hidden_units):
    input_size = input_h * input_w * channels
    weights = input_size * hidden_units
    biases = hidden_units
    return weights + biases

def conv_layer_params(kernel_h, kernel_w, in_channels, num_filters):
    weights = kernel_h * kernel_w * in_channels * num_filters
    biases = num_filters
    return weights + biases

for size in [(224, 224), (448, 448)]:
    h, w = size
    fc = fc_layer_params(h, w, 3, 1024)
    conv = conv_layer_params(3, 3, 3, 32)
    print(f"{h}x{w}x3 image -> FC first layer (1024 units): {fc:,} params")
    print(f"{h}x{w}x3 image -> Conv layer (32 filters, 3x3): {conv:,} params")
    print(f"Ratio: FC uses {fc / conv:,.0f}x more parameters than the conv layer\n")


224x224x3 image -> FC first layer (1024 units): 154,141,696 params
224x224x3 image -> Conv layer (32 filters, 3x3): 896 params
Ratio: FC uses 172,033x more parameters than the conv layer

448x448x3 image -> FC first layer (1024 units): 616,563,712 params
448x448x3 image -> Conv layer (32 filters, 3x3): 896 params
Ratio: FC uses 688,129x more parameters than the conv layer



**Why this matters in production:** 154M+ parameters in a single layer means gigabytes of memory, slower training, more GPU/edge-device compute, and — critically for a dataset of only 50,000 images — a much higher risk of overfitting, since the model has vastly more capacity to memorize training images than the data can meaningfully constrain. A CNN with comparable representational depth typically has orders of magnitude fewer total parameters (millions, not billions), trains faster, fits on modest inference hardware for real-time line deployment, and generalizes better because the parameter count is tied to feature complexity, not raw pixel count.

**Bottom line:** for a 224×224×3 image, an FC network's first layer alone can dwarf an entire CNN's parameter budget — the gap only grows as image resolution increases.

## 2. Parameter Sharing and Local Connectivity

Two structural choices in CNNs are what drive the efficiency gap above:

- **Local connectivity:** Instead of connecting every input pixel to every neuron, each convolutional neuron only looks at a small local patch (e.g., 3×3 pixels). This mirrors how defects actually manifest — a crack, dent, or finish flaw is a *local* pattern in a small neighborhood of pixels, not something defined by a relationship between a pixel in the top-left corner and one in the bottom-right. Restricting connectivity to local regions means the network isn't wasting parameters modeling irrelevant long-range pixel relationships.
- **Parameter sharing:** The *same* filter (the same 3×3 set of weights) is reused — slid — across every position in the image. This is the direct analog of biological vision: the visual cortex doesn't dedicate a separate "edge detector" to every possible retinal location; the same basic detectors are applied across the visual field, and position is handled by *which* neurons fire, not by having distinct weights per location. Practically, this means a filter that learns to detect a hairline crack pattern can detect that same crack whether it appears near a bolt hole in the top-left of the part or along an edge in the bottom-right — the network doesn't need to separately re-learn the concept for every possible image location.

Together these two ideas directly attack the **curse of dimensionality**: an FC network's parameter count grows with the *product* of input size and hidden units, so as images get larger the number of parameters needed to cover every possible pixel-to-neuron connection explodes combinatorially. Local connectivity shrinks the *receptive field* each neuron needs to consider, and parameter sharing means that shrunken filter is defined *once* and reused everywhere, rather than needing a distinct copy per location. The result is a parameter count that scales with the complexity of the *features* being detected, not with the size of the *image* — exactly what makes it feasible to train on 224×224 (or larger) production-line images with a 50,000-image dataset.

**Bottom line:** local connectivity focuses each filter on a small, relevant neighborhood; parameter sharing lets that filter's learned pattern apply everywhere in the image at no extra parameter cost — together they make CNNs both efficient and naturally suited to detecting localized visual defects.

## 3. Spatial Information Preservation

A fully-connected network's first step is to **flatten** the image into a single long vector of pixel values. Flattening discards the 2D (or 3D, with channels) structure of the image entirely — the network sees a list of numbers with no inherent notion that pixel 500 is directly above pixel 724, or that a cluster of pixels forms a contiguous region. Any spatial relationship the network exploits has to be *re-learned* indirectly through training, inefficiently, and is easily disrupted if a defect shifts position in the frame.

CNNs never flatten the image until much later (if at all, before a final classification head). Convolutional and pooling layers preserve the 2D grid structure throughout the network — the output of a conv layer is itself a spatial feature map, where each location still corresponds to a region of the original image. This has two direct consequences for our inspection system:

- **Robustness to position:** A crack near the left edge of a bracket and the same crack near the right edge produce a similar activation pattern in the corresponding (shifted) location of the feature map, rather than looking like a completely different input vector, as it would to an FC network.
- **Enabling localization, not just classification:** Because spatial structure survives through the network, CNN-based architectures can be extended (with minimal changes) to point to *where* a defect is — bounding boxes (object detection) or pixel-level defect maps (segmentation) — not just whether one exists. An FC network that has already flattened and mixed all spatial information has no such structure left to recover; it can really only answer yes/no questions, not "show me where."

**Bottom line:** CNNs keep the image's 2D layout intact as data flows through the network, which both makes learning more sample-efficient and unlocks localization/segmentation capabilities that a flattened FC representation structurally cannot support.

## 4. Where This Breaks an FC Network in Practice

Consider a concrete inspection scenario: **detecting a small stress crack that can appear anywhere on a metal bracket**, and the bracket itself may be slightly rotated or shifted between shots because parts aren't perfectly aligned on the conveyor.

- **FC network:** During training, the network sees cracks at pixel positions corresponding to wherever they happened to occur in the training images. Because FC weights are position-specific (a weight connects one *specific* input pixel to one *specific* neuron), a crack appearing in a new position at inference time activates a completely different set of weights than the ones trained on cracks elsewhere — weights the network may never have learned to associate with "defect." To generalize across all possible crack positions, an FC network would need to see (or effectively memorize) crack examples at every plausible location, which our 50,000-image dataset cannot exhaustively cover, and would still need enormous capacity (per the parameter math above) to do so.
- **CNN:** Because filters are shared across all spatial positions, a filter that learns "this pattern = crack" during training on some images fires just as strongly wherever that pattern appears in a new image, regardless of position. Pooling layers add a further degree of tolerance to small shifts. The network effectively learns the *concept* of a crack once and applies it everywhere, rather than learning "crack-at-position-X" as a distinct case for every X.

This is precisely the gap between the two architectures for our task: parts move, rotate slightly, and get photographed at varying positions on the line, while defects are inherently local and position-independent phenomena. A CNN's parameter sharing and local connectivity are a direct architectural match for that reality; an FC network's position-specific weights are fundamentally mismatched to it.

**Bottom line:** for image-based defect detection with position variability and limited labeled data, CNNs aren't just more efficient than FC networks — they encode the right assumptions (locality, translation invariance) that FC networks have to try to learn from scratch, and likely can't fully learn from 50,000 images alone.